In [ ]:
!pip install -q transformers datasets sentencepiece accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import gc
from pathlib import Path
from sklearn.model_selection import train_test_split,KFold
from datasets import Dataset
import re
import ast

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5Tokenizer,
    T5ForConditionalGeneration,
    set_seed
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

def clear_memory():
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

Device: cuda


In [ ]:
def clean_pred(text):
  if not isinstance(text, str):
    return ""

  text = text.replace("<extra_id_0>", "")
  text = text.replace("<extra_id_1>", "")
  text = text.replace("<pad>", "")
  text = text.replace("</s>", "")
  text = " ".join(text.split())

  return text.strip()


def parse_target_string(text):
  if not isinstance(text, str):
    return set()

  text = clean_pred(text)

  quads = []

  for part in text.split("[SSEP]"):
      part = part.strip()

      if not part:
        continue

      try:
        a = part.split("[A]")[1].split("[C]")[0].strip()
        c = part.split("[C]")[1].split("[O]")[0].strip()
        o = part.split("[O]")[1].split("[P]")[0].strip()
        p = part.split("[P]")[1].strip()

        quads.append((a, c, o, p))

      except Exception:
        continue

  return set(quads)


def evaluate_asqp(golds, preds):
  total_correct = 0
  total_pred = 0
  total_gold = 0

  for gold, pred in zip(golds, preds):
    gold_set = parse_target_string(gold)
    pred_set = parse_target_string(pred)

    total_correct += len(gold_set & pred_set)
    total_pred += len(pred_set)
    total_gold += len(gold_set)

  precision = total_correct / total_pred if total_pred else 0
  recall = total_correct / total_gold if total_gold else 0

  f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
  )

  return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "correct": total_correct,
        "pred_total": total_pred,
        "gold_total": total_gold
    }

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
def clean_text(text):
  if pd.isna(text):
    return ""

  text = str(text)
  text = text.lower()
  text = re.sub(r"[^a-záàâãéêíóôõúç\s]", " ", text)
  text = re.sub(r"\d+", " ", text)
  text = re.sub(r"\s+", " ", text).strip()
  return text


def clean_phrase(text):
  if text is None:
    return "NULL"

  if isinstance(text, float) and pd.isna(text):
    return "NULL"

  text = clean_text(text)

  if text == "":return "NULL"

  return text

In [ ]:
def parse_quadruples(x):
  if pd.isna(x):
    return []

  if isinstance(x, list):
    return x

  x = str(x).strip()

  if x == "" or x == "[]":
    return []

  try:
    parsed = ast.literal_eval(x)

    if isinstance(parsed, list):
      return parsed

    return []

  except Exception as e:
      print("Erro ao converter target_quadruples:")
      print(x[:500])
      print("Erro:", e)
      return []


def target_from_quads(quads):
  targets = []

  for q in quads:
    if not isinstance(q, dict):
      continue

    aspect = q.get("aspect", {})
    sentiment = q.get("sentiment", {})

    if isinstance(aspect, dict):
      aspect_term = clean_phrase(aspect.get("term", "NULL"))
    else:
      aspect_term = clean_phrase(aspect)

    category = clean_phrase(q.get("category", "NULL"))

    if isinstance(sentiment, dict):
      opinion_term = clean_phrase(sentiment.get("term", "NULL"))
    else:
      opinion_term = clean_phrase(sentiment)

    polarity = clean_phrase(q.get("polarity", "NULL"))

    targets.append(f"[A] {aspect_term} [C] {category} [O] {opinion_term} [P] {polarity}")

  return " [SSEP] ".join(targets)

In [ ]:
TRAIN_PATH = "/content/drive/MyDrive/ASQP/data_processado/aug_sr.csv"
TEST_PATH = "/content/drive/MyDrive/ASQP/data_processado/test_processed.csv"
FINAL_OUT_DIR = Path("/content/drive/MyDrive/ASQP/final_raw")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

In [ ]:
def generate_predictions(model, tokenizer, texts):
  model.eval()
  preds = []

  for text in texts:
    inputs = tokenizer(
          text,
          return_tensors="pt",
          max_length=MAX_INPUT_LEN,
          truncation=True
    ).to(model.device)

    with torch.no_grad():
      outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LEN,
                num_beams=4,
                early_stopping=True)

      pred = tokenizer.decode(outputs[0], skip_special_tokens=False)
      pred = clean_pred(pred)

      preds.append(pred)

  return preds

In [ ]:
def tokenize_batch(batch):
  model_inputs = tokenizer(
        batch["input"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False
  )

  labels = tokenizer(
        batch["target"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False
  )

  model_inputs["labels"] = labels["input_ids"]

  return model_inputs

In [ ]:
BEST_PARAMS = {
    "google/mt5-base": {
        "learning_rate": 0.00037755974095174426,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "num_train_epochs": 6
    },

    "unicamp-dl/ptt5-base-portuguese-vocab": {
        "learning_rate": 0.00045105123614691615,
        "weight_decay": 0.0,
        "warmup_ratio": 0.1,
        "num_train_epochs": 6
    }
}

In [ ]:
MODEL_NAME = "google/mt5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

SPECIAL_TOKENS = ["[A]", "[C]", "[O]", "[P]", "[SSEP]"]
tokenizer.add_tokens(SPECIAL_TOKENS)

print("Tokenizer carregado.")
print("Tamanho do vocabulário:", len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Tokenizer carregado.
Tamanho do vocabulário: 250105


In [ ]:
input_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in train_df["input"]
]

target_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in train_df["target"]
]

MAX_INPUT_LEN = int(np.percentile(input_lengths, 99))
MAX_TARGET_LEN = int(np.percentile(target_lengths, 99))

MAX_INPUT_LEN = max(MAX_INPUT_LEN, 64)
MAX_TARGET_LEN = max(MAX_TARGET_LEN, 64)

print("MAX_INPUT_LEN:", MAX_INPUT_LEN)
print("MAX_TARGET_LEN:", MAX_TARGET_LEN)

MAX_INPUT_LEN: 382
MAX_TARGET_LEN: 152


In [ ]:
def final_finetune_and_evaluate(model_name,tokenizer,train_df,test_df,FINAL_OUT_DIR):
  clear_memory()

  print("=" * 100)
  print("Fine-tuning final:", model_name)
  print("=" * 100)

  params = BEST_PARAMS[model_name]

  model_safe_name = model_name.replace("/", "_")
  output_dir = Path(FINAL_OUT_DIR) / model_safe_name
  output_dir.mkdir(parents=True, exist_ok=True)

# =====================================================
# Dataset de treino
# =====================================================

  train_data = train_df.copy().reset_index(drop=True)

  train_data = train_data.dropna(
        subset=["input", "target"]
  ).reset_index(drop=True)

  train_data["input"] = train_data["input"].astype(str)
  train_data["target"] = train_data["target"].astype(str)

  train_dataset = Dataset.from_pandas(train_data)

  train_tokenized = train_dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=train_dataset.column_names
  )


# =====================================================
# Modelo
# =====================================================

  if model_name == "unicamp-dl/ptt5-base-portuguese-vocab":
    model = T5ForConditionalGeneration.from_pretrained(
            "unicamp-dl/ptt5-base-portuguese-vocab")
  else:
    model = AutoModelForSeq2SeqLM.from_pretrained(
          model_name,
          tie_word_embeddings=False)



  model.resize_token_embeddings(len(tokenizer))
  model.to(DEVICE)

  data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=-100
  )

# =====================================================
# Fine-tuning final
# =====================================================

  training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir / "checkpoints"),

        learning_rate=params["learning_rate"],
        num_train_epochs=params["num_train_epochs"],

        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=1,

        weight_decay=params["weight_decay"],
        warmup_ratio=params["warmup_ratio"],

        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LEN,

        save_strategy="no",
        logging_strategy="epoch",

        fp16=False,
        report_to="none",

        seed=SEED,
        data_seed=SEED
  )

  trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        data_collator=data_collator
  )

  trainer.train()

# =====================================================
# Salvar modelo
# =====================================================

  model_save_dir = output_dir / "model"

  trainer.save_model(model_save_dir)
  tokenizer.save_pretrained(model_save_dir)

  print("Modelo salvo em:", model_save_dir)

# =====================================================
# Inferência no teste
# =====================================================

  test_data = test_df.copy().reset_index(drop=True)

  test_data = test_data.dropna(
        subset=["input", "target"]
  ).reset_index(drop=True)

  test_data["input"] = test_data["input"].astype(str)
  test_data["target"] = test_data["target"].astype(str)

  preds = generate_predictions(
        model=model,
        tokenizer=tokenizer,
        texts=test_data["input"].tolist()
  )

  golds = test_data["target"].tolist()

# =====================================================
# Avaliação
# =====================================================

  metrics = evaluate_asqp(golds, preds)

  print("Métricas finais:")
  print(metrics)

# =====================================================
# Salvar resultados
# =====================================================

  pred_df = pd.DataFrame({
        "input": test_data["input"].tolist(),
        "gold": golds,
        "prediction": preds})

  if "text" in test_data.columns:
    pred_df.insert(0,"text",test_data["text"].astype(str).tolist())

  pred_path = output_dir / "predictions_test_mt5_sr.csv"
  metrics_path = output_dir / "metrics_test_mt5_sr.json"
  config_path = output_dir / "final_config_mt5_sr.json"

  pred_df.to_csv(
        pred_path,
        index=False,
        encoding="utf-8"
  )

  with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
            metrics,
            f,
            ensure_ascii=False,
            indent=2
    )

  with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
            {
                "model_name": model_name,
                "best_params": params,
                "max_input_len": MAX_INPUT_LEN,
                "max_target_len": MAX_TARGET_LEN,
                "train_size": len(train_data),
                "test_size": len(test_data)
            },
            f,
            ensure_ascii=False,
            indent=2
    )

  print("Predições salvas em:", pred_path)
  print("Métricas salvas em:", metrics_path)
  print("Config salva em:", config_path)

  del model
  del trainer
  del train_tokenized
  clear_memory()

  return metrics, pred_df

# Fine Tune and Inference - mt5

In [ ]:
MODEL_NAME = "google/mt5-base"

metrics_mt5, pred_mt5 = final_finetune_and_evaluate(
    model_name=MODEL_NAME,
    tokenizer=tokenizer,
    train_df=train_df,
    test_df=test_df,
    FINAL_OUT_DIR=FINAL_OUT_DIR)

Fine-tuning final: google/mt5-base


Map:   0%|          | 0/2190 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
548,6.288702
1096,0.415460
1644,0.249291
2192,0.159388
2740,0.102548
3288,0.073341


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em: /content/drive/MyDrive/ASQP/final_raw/google_mt5-base/model
Métricas finais:
{'precision': 0.5162907268170426, 'recall': 0.5264054514480409, 'f1': 0.5212990299451707, 'correct': 618, 'pred_total': 1197, 'gold_total': 1174}
Predições salvas em: /content/drive/MyDrive/ASQP/final_raw/google_mt5-base/predictions_test_mt5_sr.csv
Métricas salvas em: /content/drive/MyDrive/ASQP/final_raw/google_mt5-base/metrics_test_mt5_sr.json
Config salva em: /content/drive/MyDrive/ASQP/final_raw/google_mt5-base/final_config_mt5_sr.json


# Fine Tune and Inference - ptt5

In [ ]:
MODEL_NAME = "unicamp-dl/ptt5-base-portuguese-vocab"

SPECIAL_TOKENS = ["[A]", "[C]", "[O]", "[P]", "[SSEP]"]

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_tokens(SPECIAL_TOKENS)


print("Tokenizer carregado.")
print("Tamanho do vocabulário:", len(tokenizer))

spiece.model:   0%|          | 0.00/756k [00:00<?, ?B/s]

Tokenizer carregado.
Tamanho do vocabulário: 32105


In [ ]:
input_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in train_df["input"]
]

target_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in train_df["target"]
]

MAX_INPUT_LEN = int(np.percentile(input_lengths, 99))
MAX_TARGET_LEN = int(np.percentile(target_lengths, 99))

MAX_INPUT_LEN = max(MAX_INPUT_LEN, 64)
MAX_TARGET_LEN = max(MAX_TARGET_LEN, 64)

print("MAX_INPUT_LEN:", MAX_INPUT_LEN)
print("MAX_TARGET_LEN:", MAX_TARGET_LEN)

MAX_INPUT_LEN: 308
MAX_TARGET_LEN: 171


In [ ]:
MODEL_NAME = "unicamp-dl/ptt5-base-portuguese-vocab"

metrics_ptt5, pred_ptt5 = final_finetune_and_evaluate(
    model_name=MODEL_NAME,
    tokenizer=tokenizer,
    train_df=train_df,
    test_df=test_df,
    FINAL_OUT_DIR=FINAL_OUT_DIR)

Fine-tuning final: unicamp-dl/ptt5-base-portuguese-vocab


Map:   0%|          | 0/2190 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
548,3.561700
1096,0.423290
1644,0.220802
2192,0.132380
2740,0.077948
3288,0.050262


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em: /content/drive/MyDrive/ASQP/final_raw/unicamp-dl_ptt5-base-portuguese-vocab/model
Métricas finais:
{'precision': 0.5297157622739018, 'recall': 0.5238500851788757, 'f1': 0.5267665952890792, 'correct': 615, 'pred_total': 1161, 'gold_total': 1174}
Predições salvas em: /content/drive/MyDrive/ASQP/final_raw/unicamp-dl_ptt5-base-portuguese-vocab/predictions_test_mt5_sr.csv
Métricas salvas em: /content/drive/MyDrive/ASQP/final_raw/unicamp-dl_ptt5-base-portuguese-vocab/metrics_test_mt5_sr.json
Config salva em: /content/drive/MyDrive/ASQP/final_raw/unicamp-dl_ptt5-base-portuguese-vocab/final_config_mt5_sr.json
